# 05 Chart And Signal Scanner

The scanner is for visual signal inspection on charts. It is not a full backtest: it does not simulate lot sizing, commission, swap, or pending fills like the execution engine.

Notebook goals:

- Inspect one symbol on a chart.
- Scan multiple symbols for currently interesting setups.
- Summarize pass/rejected/win/outcome counts in a readable way.


In [ ]:
#
#

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Could not find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:

import pandas as pd
from IPython.display import display

from core_python.strategies.combo.params import SYMBOLS, get_indicator_params, summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    configure_notebook,
    plot_scan_summary,
    show_note,
    show_run_config,
)
from core_python.strategies.combo.scanner import (
    build_reversal_figure,
    calc_reversal_stats,
    run_multi_reversal_scan,
    run_reversal_scan,
)

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
#

RUN_CONFIG = {
    'scan_symbol': 'US30',
    'scan_symbols': ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD'],
    'n_bars': 250,
    'scan_params_overrides': {
        # 'MA_PERIOD': 20,
        # 'KTP': 2.3,
        # 'MIN_RR': 1.25,
    },
}

SCAN_PARAMS = get_indicator_params()
SCAN_PARAMS.update(RUN_CONFIG['scan_params_overrides'])
show_run_config('Scanner Configuration', RUN_CONFIG)
show_note('Active SCAN_PARAMS', 'These are the indicator/signal parameters used by the scanner.')
display(SCAN_PARAMS)


In [ ]:
#

single = run_reversal_scan(RUN_CONFIG['scan_symbol'], RUN_CONFIG['n_bars'], SCAN_PARAMS)
single_stats = calc_reversal_stats(single['signals_df'])
show_note('Single-Symbol Scanner Stats', 'Quick statistics for the selected symbol.')
display(pd.DataFrame([single_stats]).T.rename(columns={0: 'value'}))
display(single['signals_df'].tail(30))


In [ ]:
#

fig = build_reversal_figure(RUN_CONFIG['scan_symbol'], single, SCAN_PARAMS)
fig.show()


In [ ]:
#

multi = run_multi_reversal_scan(RUN_CONFIG['scan_symbols'], RUN_CONFIG['n_bars'], SCAN_PARAMS)
summary_rows = []
for sym, scan_result in multi.items():
    summary_rows.append({'symbol': sym, **calc_reversal_stats(scan_result['signals_df'])})

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    sort_cols = [c for c in ['n_pass', 'win_pct', 'avg_rr'] if c in summary_df.columns]
    summary_df = summary_df.sort_values(sort_cols, ascending=False, ignore_index=True)
    display(summary_df)
    plot_scan_summary(summary_df)
else:
    print('No scanner summary is available.')


In [ ]:
#

selected = summary_df.iloc[0]['symbol'] if not summary_df.empty else RUN_CONFIG['scan_symbol']
print('Best current scanner candidate =', selected)
selected_result = multi.get(selected) or single
selected_fig = build_reversal_figure(selected, selected_result, SCAN_PARAMS)
selected_fig.show()
